In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated")


Authenticated


In [ ]:
import os, time
from google.colab import drive

# 1) Drive'ı temiz şekilde bağla (mount hatalarını bitirir)
drive.mount('/content/drive', force_remount=True)

# 2) Her deneme için yeni bir klasör aç (overwrite riskini sıfırlar)
run_id = time.strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"/content/drive/MyDrive/Embeddings_clean_run_{run_id}"
os.makedirs(BASE_DIR, exist_ok=True)

print("✅ Drive mounted.")
print("✅ Run folder:", BASE_DIR)


Mounted at /content/drive
✅ Drive mounted.
✅ Run folder: /content/drive/MyDrive/Embeddings_clean_run_20260116_132610


In [ ]:
!pip install google-cloud-bigquery


In [ ]:
from google.cloud import bigquery
import pandas as pd

project_id = "dat-analytics-eng-ec869189"
client = bigquery.Client(project=project_id)

query = """
SELECT
  snid,
  user_text,
  subject_names
FROM `dat-analytics-eng-ec869189.subject_based_segmentation.training_data_3`
WHERE user_text IS NOT NULL
ORDER BY snid
"""

df = client.query(query).to_dataframe()

print("✅ Rows:", df.shape[0])
print("✅ Columns:", df.columns.tolist())
print("Example snid:", df.iloc[0]["snid"])
print("Example text:", df.iloc[0]["user_text"][:120])


KeyboardInterrupt: 

In [ ]:
Big Query - take the snapshot of df for safety

In [ ]:
SNAPSHOT_PATH = f"{BASE_DIR}/training_df.parquet"

df.to_parquet(SNAPSHOT_PATH, index=False)

print("✅ Snapshot saved:", SNAPSHOT_PATH)
print("Rows:", len(df))


NameError: name 'df' is not defined

In [ ]:
Lock this order- save the snid list and row count to drive

In [ ]:
import numpy as np
import os

# run klasörün BASE_DIR değişkeni hala duruyorsa kullanacağız.
# (Eğer BASE_DIR tanımlı değilse söyle, tek satırda tekrar tanımlarız.)

snid_path = os.path.join(BASE_DIR, "snids_ordered.npy")
count_path = os.path.join(BASE_DIR, "row_count.txt")

snids = df["snid"].values  # ORDER BY snid ile gelen kesin sıra

np.save(snid_path, snids)

with open(count_path, "w") as f:
    f.write(str(len(snids)))

print("✅ Saved snids:", snid_path)
print("✅ Rows:", len(snids))
print("✅ First snid:", snids[0])
print("✅ Last snid:", snids[-1])


✅ Saved snids: /content/drive/MyDrive/Embeddings_clean_run_20260116_112012/snids_ordered.npy
✅ Rows: 113379
✅ First snid: 000073f5-896a-4a37-b1e3-2cd95d429040_SN
✅ Last snid: fffdf35f-3510-4851-a7e1-65d9e292175d_SN


labels

In [ ]:
labels = (
    df["subject_names"]
      .fillna("")
      .astype(str)
      .str.split(",")
      .apply(lambda xs: [x.strip() for x in xs if x.strip() != ""])
      .tolist()
)


NameError: name 'df' is not defined

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import os


# Multi-label binarizer
mlb = MultiLabelBinarizer(sparse_output=True)
Y = mlb.fit_transform(labels)

print("✅ Label matrix shape:", Y.shape)
print("✅ Number of unique labels:", len(mlb.classes_))
print("✅ Example labels:", mlb.classes_[:10])

# kaydet (çok önemli)
np.save(os.path.join(BASE_DIR, "labels_multi_hot.npy"), Y)
np.save(os.path.join(BASE_DIR, "label_classes.npy"), mlb.classes_)


✅ Label matrix shape: (113379, 46)
✅ Number of unique labels: 46
✅ Example labels: ['2866' '2869' '2872' '2873' '2874' '2878' '2879' '2882' '2886' '2887']


Embeddings

In [ ]:
import os

EMB_DIR = os.path.join(BASE_DIR, "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

print("✅ Embedding dir:", EMB_DIR)
print("Existing batches:", len(os.listdir(EMB_DIR)))


✅ Embedding dir: /content/drive/MyDrive/Embeddings_clean_run_20260116_112012/embeddings
Existing batches: 0


In [ ]:
!pip -q install -U "huggingface_hub==0.34.6" "transformers==4.41.2" "sentence-transformers==3.0.1" "pyarrow==16.0.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 14.1 MB/s eta 0:00:00


In [ ]:
import os, math, numpy as np
from sentence_transformers import SentenceTransformer

# 1) Model (GPU varsa otomatik kullanır)
model = SentenceTransformer("all-mpnet-base-v2")

# 2) Batch ayarları
batch_size = 128              # dosyaya yazacağımız ana batch (disk batch)
encode_mini_batch = 64        # model.encode içindeki mini-batch (T4 için güvenli)

# 3) Kayıt klasörü (senin daha önce açtığın klasör)
EMB_DIR = os.path.join(BASE_DIR, "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

# 4) Kaldığı yeri bul (klasörde kaç batch varsa oradan devam)
existing = sorted([f for f in os.listdir(EMB_DIR) if f.startswith("batch_") and f.endswith(".npy")])
start_batch = len(existing)

total_batches = math.ceil(len(df) / batch_size)

print("✅ EMB_DIR:", EMB_DIR)
print("✅ df rows:", len(df))
print("✅ batch_size:", batch_size, "| encode_mini_batch:", encode_mini_batch)
print("✅ existing batches:", start_batch)
print("✅ total_batches:", total_batches)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ EMB_DIR: /content/drive/MyDrive/Embeddings_clean_run_20260116_112012/embeddings
✅ df rows: 113379
✅ batch_size: 128 | encode_mini_batch: 64
✅ existing batches: 0
✅ total_batches: 886


In [ ]:
# 6) Embedding üretimi (batch batch, güvenli, restart-proof)

for b in range(start_batch, total_batches):
    start = b * batch_size
    end = min((b + 1) * batch_size, len(df))

    texts = df["user_text"].iloc[start:end].tolist()

    emb = model.encode(
        texts,
        batch_size=encode_mini_batch,   # GPU mini-batch
        show_progress_bar=False,
        normalize_embeddings=False
    ).astype("float32")

    out_path = os.path.join(EMB_DIR, f"batch_{b:05d}.npy")
    np.save(out_path, emb)

    # kontroll logu (çok sık yazdırmıyoruz)
    if (b % 10 == 0) or (b == total_batches - 1):
        print(
            f"✅ Saved batch {b}/{total_batches-1} | "
            f"rows {start}:{end} | shape={emb.shape}"
        )


✅ Saved batch 0/885 | rows 0:128 | shape=(128, 768)
✅ Saved batch 10/885 | rows 1280:1408 | shape=(128, 768)
✅ Saved batch 20/885 | rows 2560:2688 | shape=(128, 768)
✅ Saved batch 30/885 | rows 3840:3968 | shape=(128, 768)
✅ Saved batch 40/885 | rows 5120:5248 | shape=(128, 768)
✅ Saved batch 50/885 | rows 6400:6528 | shape=(128, 768)
✅ Saved batch 60/885 | rows 7680:7808 | shape=(128, 768)
✅ Saved batch 70/885 | rows 8960:9088 | shape=(128, 768)
✅ Saved batch 80/885 | rows 10240:10368 | shape=(128, 768)
✅ Saved batch 90/885 | rows 11520:11648 | shape=(128, 768)
✅ Saved batch 100/885 | rows 12800:12928 | shape=(128, 768)
✅ Saved batch 110/885 | rows 14080:14208 | shape=(128, 768)
✅ Saved batch 120/885 | rows 15360:15488 | shape=(128, 768)
✅ Saved batch 130/885 | rows 16640:16768 | shape=(128, 768)
✅ Saved batch 140/885 | rows 17920:18048 | shape=(128, 768)
✅ Saved batch 150/885 | rows 19200:19328 | shape=(128, 768)
✅ Saved batch 160/885 | rows 20480:20608 | shape=(128, 768)
✅ Saved bat

IF it disconnects:

In [ ]:
import os

files = sorted([f for f in os.listdir(EMB_DIR) if f.startswith("batch_") and f.endswith(".npy")])
print("Saved files:", len(files))
print("Last file:", files[-1] if files else None)


Saved files: 886
Last file: batch_00885.npy


Find the location of embeddings

In [ ]:
import os, glob

# Drive'daki tüm "Embeddings_clean_run_*" klasörlerini bul
runs = sorted(glob.glob("/content/drive/MyDrive/Embeddings_clean_run_*"))

print("Found runs:", len(runs))
for r in runs[-10:]:  # son 10 run
    emb_dir = os.path.join(r, "embeddings")
    n = len([f for f in os.listdir(emb_dir) if f.startswith("batch_") and f.endswith(".npy")]) if os.path.exists(emb_dir) else 0
    print(os.path.basename(r), "-> batches:", n, "| path:", emb_dir)


Found runs: 11
Embeddings_clean_run_20260114_101402 -> batches: 0 | path: /content/drive/MyDrive/Embeddings_clean_run_20260114_101402/embeddings
Embeddings_clean_run_20260114_105553 -> batches: 0 | path: /content/drive/MyDrive/Embeddings_clean_run_20260114_105553/embeddings
Embeddings_clean_run_20260114_110139 -> batches: 0 | path: /content/drive/MyDrive/Embeddings_clean_run_20260114_110139/embeddings
Embeddings_clean_run_20260114_153923 -> batches: 2578 | path: /content/drive/MyDrive/Embeddings_clean_run_20260114_153923/embeddings
Embeddings_clean_run_20260115_085803 -> batches: 0 | path: /content/drive/MyDrive/Embeddings_clean_run_20260115_085803/embeddings
Embeddings_clean_run_20260115_114612 -> batches: 0 | path: /content/drive/MyDrive/Embeddings_clean_run_20260115_114612/embeddings
Embeddings_clean_run_20260115_134536 -> batches: 0 | path: /content/drive/MyDrive/Embeddings_clean_run_20260115_134536/embeddings
Embeddings_clean_run_20260115_191227 -> batches: 62 | path: /content/dri

Confirm the current batches

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260114_153923"
EMB_DIR = os.path.join(BASE_DIR, "embeddings")

files = sorted([f for f in os.listdir(EMB_DIR) if f.startswith("batch_") and f.endswith(".npy")])

print("✅ BASE_DIR:", BASE_DIR)
print("✅ EMB_DIR:", EMB_DIR)
print("✅ existing batches:", len(files))
print("✅ last batch file:", files[-1] if files else None)


✅ BASE_DIR: /content/drive/MyDrive/Embeddings_clean_run_20260114_153923
✅ EMB_DIR: /content/drive/MyDrive/Embeddings_clean_run_20260114_153923/embeddings
✅ existing batches: 1731
✅ last batch file: batch_01730.npy


Don't run the df again, load the snapshot of df

In [ ]:
import pandas as pd
import os

SNAPSHOT_PATH = os.path.join(BASE_DIR, "training_df.parquet")
df = pd.read_parquet(SNAPSHOT_PATH)

print("✅ df loaded:", df.shape)
print("✅ first snid:", df.iloc[0]["snid"])
print("✅ last snid:", df.iloc[-1]["snid"])
print("✅ example text:", str(df.iloc[0]["user_text"])[:120])


✅ df loaded: (659785, 3)
✅ first snid: 000049b2-7235-4e24-b350-81917dad93db_SN
✅ last snid: ffffcdb0-e28a-444a-ae2b-4c03e3d431b4_SN
✅ example text: Chemical and Biological Technologies in Agriculture Scientific Reports Applied Biochemistry and Biotechnology Egyptian J


Continue producing the embeddings in right order

In [ ]:
import os, math, numpy as np
from sentence_transformers import SentenceTransformer

# ✅ 1)right folder
BASE_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260114_153923"
EMB_DIR = os.path.join(BASE_DIR, "embeddings")

# ✅ 2) Model
model = SentenceTransformer("all-mpnet-base-v2")

# ✅ 3) settings
batch_size = 256
encode_mini_batch = 64

# ✅ 4) where were we?
existing = sorted([f for f in os.listdir(EMB_DIR) if f.startswith("batch_") and f.endswith(".npy")])
start_batch = len(existing)            # 1731 bekliyoruz
total_batches = math.ceil(len(df) / batch_size)

print("✅ existing batches:", start_batch, "| total:", total_batches)

# ✅ 5) continue
for b in range(start_batch, total_batches):
    start = b * batch_size
    end = min((b + 1) * batch_size, len(df))

    texts = df["user_text"].iloc[start:end].tolist()

    emb = model.encode(
        texts,
        batch_size=encode_mini_batch,
        show_progress_bar=False,
        normalize_embeddings=False
    ).astype("float32")

    out_path = os.path.join(EMB_DIR, f"batch_{b:05d}.npy")
    np.save(out_path, emb)

    # give log every 10 batch
    if (b % 10 == 0) or (b == total_batches - 1):
        print(f"✅ Saved batch {b}/{total_batches-1} | rows {start}:{end} | shape={emb.shape}")


✅ existing batches: 1731 | total: 2578
✅ Saved batch 1740/2577 | rows 445440:445696 | shape=(256, 768)
✅ Saved batch 1750/2577 | rows 448000:448256 | shape=(256, 768)
✅ Saved batch 1760/2577 | rows 450560:450816 | shape=(256, 768)
✅ Saved batch 1770/2577 | rows 453120:453376 | shape=(256, 768)
✅ Saved batch 1780/2577 | rows 455680:455936 | shape=(256, 768)
✅ Saved batch 1790/2577 | rows 458240:458496 | shape=(256, 768)
✅ Saved batch 1800/2577 | rows 460800:461056 | shape=(256, 768)
✅ Saved batch 1810/2577 | rows 463360:463616 | shape=(256, 768)
✅ Saved batch 1820/2577 | rows 465920:466176 | shape=(256, 768)
✅ Saved batch 1830/2577 | rows 468480:468736 | shape=(256, 768)
✅ Saved batch 1840/2577 | rows 471040:471296 | shape=(256, 768)
✅ Saved batch 1850/2577 | rows 473600:473856 | shape=(256, 768)
✅ Saved batch 1860/2577 | rows 476160:476416 | shape=(256, 768)
✅ Saved batch 1870/2577 | rows 478720:478976 | shape=(256, 768)
✅ Saved batch 1880/2577 | rows 481280:481536 | shape=(256, 768)
✅

Embeddings are created check if everything is okay and detect if something is missing

In [ ]:
import os, re

# 🔒 DOĞRU RUN KLASÖRÜ (elle sabitliyoruz)
BASE_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260114_153923"
EMB_DIR = os.path.join(BASE_DIR, "embeddings")

print("BASE_DIR exists:", os.path.exists(BASE_DIR))
print("EMB_DIR exists:", os.path.exists(EMB_DIR))


BASE_DIR exists: False
EMB_DIR exists: False


In [ ]:
import os, glob

# Önce Drive bağlı mı kontrol
print("Drive exists:", os.path.exists("/content/drive/MyDrive"))

# Tüm run klasörlerini bul
runs = sorted(glob.glob("/content/drive/MyDrive/Embeddings_clean_run_*"))

print("Found runs:", len(runs))

for r in runs:
    emb_dir = os.path.join(r, "embeddings")
    if os.path.exists(emb_dir):
        batches = len([f for f in os.listdir(emb_dir) if f.startswith("batch_")])
        print(os.path.basename(r), "-> batches:", batches)
    else:
        print(os.path.basename(r), "-> NO embeddings folder")


Drive exists: True
Found runs: 11
Embeddings_clean_run_20260114_092555 -> NO embeddings folder
Embeddings_clean_run_20260114_101402 -> batches: 0
Embeddings_clean_run_20260114_105553 -> batches: 0
Embeddings_clean_run_20260114_110139 -> batches: 0
Embeddings_clean_run_20260114_153923 -> batches: 2578
Embeddings_clean_run_20260115_085803 -> batches: 0
Embeddings_clean_run_20260115_114612 -> NO embeddings folder
Embeddings_clean_run_20260115_134536 -> NO embeddings folder
Embeddings_clean_run_20260115_191227 -> batches: 62
Embeddings_clean_run_20260116_105048 -> batches: 40
Embeddings_clean_run_20260116_112012 -> batches: 886


In [ ]:
import os, re, numpy as np

# Embedding batch dosyalarını listele
files = sorted([f for f in os.listdir(EMB_DIR) if f.startswith("batch_") and f.endswith(".npy")])

print("✅ batch file count:", len(files))
print("✅ first file:", files[0] if files else None)
print("✅ last file:", files[-1] if files else None)

# Toplam satır sayısını hesapla
total_rows = 0
bad_shapes = []

for f in files:
    arr = np.load(os.path.join(EMB_DIR, f), mmap_mode="r")   # RAM şişmesin diye
    if arr.ndim != 2 or arr.shape[1] != 768:
        bad_shapes.append((f, arr.shape))
    total_rows += arr.shape[0]

print("✅ total embedded rows:", total_rows)
print("✅ df rows:", len(df))
print("✅ rows match:", total_rows == len(df))

if bad_shapes:
    print("⚠️ unexpected shapes:", bad_shapes[:10])


✅ batch file count: 2578
✅ first file: batch_00000.npy
✅ last file: batch_02577.npy
✅ total embedded rows: 659785


NameError: name 'df' is not defined

In [ ]:
import os
import pandas as pd
import numpy as np

# doğru run klasörü: sende şu an bu olmalı
BASE_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260116_112012"

SNAPSHOT_PATH = os.path.join(BASE_DIR, "training_df.parquet")
SNID_PATH = os.path.join(BASE_DIR, "snids_ordered.npy")
EMB_DIR = os.path.join(BASE_DIR, "embeddings")

df = pd.read_parquet(SNAPSHOT_PATH)
snids = np.load(SNID_PATH, allow_pickle=True)

print("✅ df shape:", df.shape)
print("✅ snids shape:", snids.shape)
print("✅ EMB_DIR exists:", os.path.exists(EMB_DIR))
print("✅ first snid df:", df.iloc[0]["snid"])
print("✅ first snid file:", snids[0])
print("✅ match first:", df.iloc[0]["snid"] == snids[0])


✅ df shape: (113379, 3)
✅ snids shape: (113379,)
✅ EMB_DIR exists: True
✅ first snid df: 000073f5-896a-4a37-b1e3-2cd95d429040_SN
✅ first snid file: 000073f5-896a-4a37-b1e3-2cd95d429040_SN
✅ match first: True


cosine similarity to check if the embedding is true

In [ ]:
import os, re
import numpy as np
from sentence_transformers import SentenceTransformer

# 1) model (embedding üretirken kullandığın aynı model olmalı)
model = SentenceTransformer("all-mpnet-base-v2")

# 2) yardımcı: dosyadan embedding okuma (index -> hangi batch, hangi satır)
batch_size = 128

def load_saved_embedding_at(index: int):
    b = index // batch_size          # batch numarası
    j = index % batch_size           # batch içindeki offset
    path = os.path.join(EMB_DIR, f"batch_{b:05d}.npy")
    arr = np.load(path, mmap_mode="r")   # RAM şişmesin
    return np.array(arr[j], dtype=np.float32), path, arr.shape

def cosine(a, b):
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# 3) test edeceğimiz index'ler (baş / orta / son + rastgele)
test_idxs = [0, 1234, 50000, len(df)-1]

for idx in test_idxs:
    text = str(df.iloc[idx]["user_text"])
    emb_now = model.encode([text], show_progress_bar=False, normalize_embeddings=False)[0].astype(np.float32)

    emb_saved, p, shape = load_saved_embedding_at(idx)
    sim = cosine(emb_saved, emb_now)

    print(f"idx={idx} | cosine={sim:.4f} | file={os.path.basename(p)} | batch_shape={shape} | text_snippet={text[:60]!r}")


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 581bd7ba-a247-4d8a-8e32-5f9324bc6d6b)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 786748a8-1208-40c8-8e37-13645c5afd2b)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: baf8ff89-ddab-4cf3-9507-3f6e0482c201)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/./config_sentence_transformers.json
Retrying in 1s [Retry 1/5].


idx=0 | cosine=1.0000 | file=batch_00000.npy | batch_shape=(128, 768) | text_snippet='2-quinazolinone; all-solid-state sodium metal batteries; alu'
idx=1234 | cosine=1.0000 | file=batch_00009.npy | batch_shape=(128, 768) | text_snippet='biomedical engineering; drug delivery; nanoparticles; tissue'
idx=50000 | cosine=1.0000 | file=batch_00390.npy | batch_shape=(128, 768) | text_snippet='ableism; abstraction; assessment; assessment accommodations;'
idx=113378 | cosine=1.0000 | file=batch_00885.npy | batch_shape=(99, 768) | text_snippet='ageing; agroecology; antibiotics; antimicrobial resistance; '


if cosine similarity is 0.99+ the embeddings are correct

It is time to ingest it into bigquery - sample test

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260116_112012"  # <- sende doğru olan
EMB_DIR = BASE_DIR + "/embeddings"


load the snapshot and labels

In [ ]:
import os
import numpy as np
import pandas as pd

SNAPSHOT_PATH = os.path.join(BASE_DIR, "training_df.parquet")
LABELS_PATH   = os.path.join(BASE_DIR, "labels_multi_hot.npy")

df = pd.read_parquet(SNAPSHOT_PATH)

Y_obj = np.load(LABELS_PATH, allow_pickle=True)
# bazen direkt sparse gelir, bazen object wrapper
Y = Y_obj.item() if hasattr(Y_obj, "item") else Y_obj

print("df:", df.shape)
print("Y type:", type(Y))


df: (113379, 3)
Y type: <class 'scipy.sparse._csr.csr_matrix'>


create a small dataframe and labels for testing

In [ ]:
from scipy import sparse

N = 5000  # küçük test

# labels -> dense (N x 46)
Y_small = Y[:N]
if sparse.issparse(Y_small):
    Y_small = Y_small.toarray()
Y_small = Y_small.astype("int64")

# embeddings -> ilk batchlerden okuyacağız (256'lık batch dosyaları)
batch_size = 128
need_batches = (N + batch_size - 1) // batch_size

emb_list = []
for b in range(need_batches):
    path = os.path.join(EMB_DIR, f"batch_{b:05d}.npy")
    emb_list.append(np.load(path))
emb_small = np.vstack(emb_list)[:N].astype("float32")

print("emb_small:", emb_small.shape)   # (5000, 768)
print("Y_small:", Y_small.shape)       # (5000, 46)

# kolon isimleri
emb_cols = [f"e_{i}" for i in range(emb_small.shape[1])]
y_cols   = [f"y_{i}" for i in range(Y_small.shape[1])]

df_out = pd.DataFrame(emb_small, columns=emb_cols)
for j, c in enumerate(y_cols):
    df_out[c] = Y_small[:, j]

df_out.insert(0, "snid", df["snid"].iloc[:N].values)

df_out.head()


emb_small: (5000, 768)
Y_small: (5000, 46)


,snid,e_0,e_1,e_2,e_3,e_4,e_5,e_6,e_7,e_8,...,y_36,y_37,y_38,y_39,y_40,y_41,y_42,y_43,y_44,y_45
0,000073f5-896a-4a37-b1e3-2cd95d429040_SN,0.040560,-0.040589,-0.009135,-0.018547,0.006103,0.027670,-0.012231,-0.010030,0.003190,...,0,0,0,0,0,0,0,0,0,0
1,00007beb-9142-4123-b755-eca93f09c032_SN,0.010520,-0.025478,-0.020929,0.039649,-0.002696,0.018326,0.018254,-0.071116,0.056713,...,0,0,1,0,0,0,0,0,0,0
2,00009e14-4259-499c-a1be-c6d129e957df_SN,0.026322,0.041438,-0.033200,-0.069007,0.004470,0.074862,0.012934,0.042161,0.061212,...,0,0,0,0,0,0,0,0,0,0
3,0000ff23-0b1c-4bbe-81ff-e60a99a18346_SN,-0.016332,0.030058,0.031538,0.011790,0.057154,0.024060,0.032659,0.089081,0.056595,...,0,0,0,0,0,0,0,0,0,0
4,0001cf2e-561f-4f78-84fa-3726257beb17_SN,-0.022621,0.050129,-0.025489,0.006778,-0.009556,0.019356,0.070292,-0.023247,0.125735,...,0,0,0,0,0,0,0,0,0,0


Load into BQ

In [ ]:
from google.cloud import bigquery

project_id = "snproject-cbb9ca25"
client = bigquery.Client(project=project_id)

dataset_id = "subject_based_segmentation"
table_id = f"{project_id}.{dataset_id}.embeddings_sample2"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(df_out, table_id, job_config=job_config)
job.result()

print("✅ Loaded to:", table_id)


✅ Loaded to: snproject-cbb9ca25.subject_based_segmentation.embeddings_sample2


find which label is which y0= which label

In [ ]:
# 1) Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

# 2) Dosya yolunu ayarla (Drive içindeki gerçek klasör yolunu buna göre düzenle)
LABEL_CLASSES_PATH = "/content/drive/MyDrive/Embeddings_clean_run_20260116_112012/label_classes.npy"

import numpy as np

# 3) label_classes.npy oku
label_classes = np.load(LABEL_CLASSES_PATH, allow_pickle=True)

print("Type:", type(label_classes))
print("Shape:", getattr(label_classes, "shape", None))

# 4) İçeriği kontrol et ve y_0'ı yazdır
# label_classes çoğu zaman 1D array/list olur: [label0_name, label1_name, ...]
if isinstance(label_classes, np.ndarray) and label_classes.ndim == 1:
    print("\nFirst 10 labels:")
    for i in range(min(10, len(label_classes))):
        print(f"y_{i} =", label_classes[i])

    print("\n✅ y_0 label name:", label_classes[0])

else:
    # Bazı projelerde dict/obj çıkar; güvenli şekilde yazdırıyoruz
    print("\nRaw content preview:")
    print(label_classes)

    # Eğer dict ise ve 0 anahtarı varsa
    if isinstance(label_classes.item(), dict):
        d = label_classes.item()
        if 0 in d:
            print("\n✅ y_0 label name:", d[0])
        elif "0" in d:
            print("\n✅ y_0 label name:", d["0"])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Type: <class 'numpy.ndarray'>
Shape: (46,)

First 10 labels:
y_0 = 2866
y_1 = 2869
y_2 = 2872
y_3 = 2873
y_4 = 2874
y_5 = 2878
y_6 = 2879
y_7 = 2882
y_8 = 2886
y_9 = 2887

✅ y_0 label name: 2866


Lets see which labels frequencies to see which labels are in risk

In [ ]:
import numpy as np

LABELS_PATH = "/content/drive/MyDrive/Embeddings_clean_run_20260114_153923/labels_multi_hot.npy"
CLASSES_PATH = "/content/drive/MyDrive/Embeddings_clean_run_20260114_153923/label_classes.npy"

# 1) Yükle (object olarak gelir)
Y_obj = np.load(LABELS_PATH, allow_pickle=True)
classes = np.load(CLASSES_PATH, allow_pickle=True)

print("Loaded Y_obj:", type(Y_obj), "shape:", getattr(Y_obj, "shape", None), "dtype:", getattr(Y_obj, "dtype", None))

# 2) Unwrap: npy içinde tek obje varsa .item() ile çıkar
Y = Y_obj.item() if isinstance(Y_obj, np.ndarray) and Y_obj.shape == () else Y_obj

print("Unwrapped Y:", type(Y), "shape:", getattr(Y, "shape", None))

# 3) Sparse matrix ise label frekanslarını çıkar
# csr_matrix.sum(axis=0) -> (1, 46) matrix döner, .A1 ile 1D array yapıyoruz
pos_counts = np.asarray(Y.sum(axis=0)).ravel().astype(int)

total = Y.shape[0]
pos_rates = pos_counts / total

idx_sorted = np.argsort(pos_counts)

print("\nBottom 10 labels (lowest positives):")
for i in idx_sorted[:10]:
    print(f"y_{i:<2} {str(classes[i]):<40} pos={pos_counts[i]:<8} rate={pos_rates[i]:.6f}")

print("\nTop 10 labels (highest positives):")
for i in idx_sorted[-10:][::-1]:
    print(f"y_{i:<2} {str(classes[i]):<40} pos={pos_counts[i]:<8} rate={pos_rates[i]:.6f}")


Loaded Y_obj: <class 'numpy.ndarray'> shape: () dtype: object
Unwrapped Y: <class 'scipy.sparse._csr.csr_matrix'> shape: (659785, 46)

Bottom 10 labels (lowest positives):
y_32 Linguistics and Literature               pos=1532     rate=0.002322
y_2  Archaeology and History                  pos=1628     rate=0.002467
y_24 Energy                                   pos=2804     rate=0.004250
y_20 Control, Robotics, Mechatronics          pos=3661     rate=0.005549
y_16 Civil engineering                        pos=4359     rate=0.006607
y_30 Law, Politics and International studies  pos=4571     rate=0.006928
y_33 Materials Engineering                    pos=4779     rate=0.007243
y_35 Mathematical and computational engineering applications pos=4983     rate=0.007552
y_7  Biological and physical engineering      pos=6902     rate=0.010461
y_1  Analysis                                 pos=7120     rate=0.010791

Top 10 labels (highest positives):
y_8  Biological sciences                      p

Modeling

In [ ]:
import os, numpy as np

EMB_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260116_112012/embeddings"
batch_files = sorted([f for f in os.listdir(EMB_DIR) if f.startswith("batch_") and f.endswith(".npy")])

print("batches:", len(batch_files), "first:", batch_files[0], "last:", batch_files[-1])

# birkaç batch shape kontrol
for f in batch_files[:3] + batch_files[-3:]:
    arr = np.load(os.path.join(EMB_DIR, f))
    print(f, arr.shape, arr.dtype)


batches: 886 first: batch_00000.npy last: batch_00885.npy
batch_00000.npy (128, 768) float32
batch_00001.npy (128, 768) float32
batch_00002.npy (128, 768) float32
batch_00883.npy (128, 768) float32
batch_00884.npy (128, 768) float32
batch_00885.npy (99, 768) float32


In [ ]:
import numpy as np, os

BASE_DIR = "/content/drive/MyDrive/Embeddings_clean_run_20260116_112012"
snids = np.load(os.path.join(BASE_DIR, "snids_ordered.npy"), allow_pickle=True)
print("snids:", len(snids))

# toplam embedding satır sayısı
total_rows = 0
for f in batch_files:
    total_rows += np.load(os.path.join(EMB_DIR, f)).shape[0]
print("embedding rows:", total_rows)


snids: 113379
embedding rows: 113379


model with ranking approach top 3 predictions

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

# 1) Train/test split (row-level; small demo için yeterli)
idx = np.arange(len(emb_small))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train, X_test = emb_small[train_idx], emb_small[test_idx]
Y_train, Y_test = Y_small[train_idx], Y_small[test_idx]

print("train:", X_train.shape, Y_train.shape)
print("test :", X_test.shape, Y_test.shape)

# 2) Model (multi-label score üretir)
clf = OneVsRestClassifier(
    LogisticRegression(
        solver="saga",
        max_iter=300,
        n_jobs=-1,
        class_weight="balanced"
    )
)

clf.fit(X_train, Y_train)

# 3) Skorlar (probabilities)
P_test = clf.predict_proba(X_test)   # shape: (n_test, 46)
print("P_test:", P_test.shape)

# 4) Top-K metrikleri (ürün hedefinle uyumlu)
def topk_metrics(y_true, y_score, k=3):
    hits_any = 0          # en az 1 doğru label top-k içinde mi?
    prec_sum = 0.0
    recall_sum = 0.0
    n_eval = 0

    for i in range(y_true.shape[0]):
        true = np.where(y_true[i] == 1)[0]
        if true.size == 0:
            continue
        n_eval += 1

        topk = np.argsort(-y_score[i])[:k]
        inter = np.intersect1d(topk, true).size

        if inter > 0:
            hits_any += 1
        prec_sum += inter / k
        recall_sum += inter / true.size

    recall_any = hits_any / n_eval if n_eval else 0.0
    precision_k = prec_sum / n_eval if n_eval else 0.0
    recall_k = recall_sum / n_eval if n_eval else 0.0

    return recall_any, precision_k, recall_k, n_eval

r_any_3, p3, r3, n_eval = topk_metrics(Y_test, P_test, k=3)
r_any_5, p5, r5, _      = topk_metrics(Y_test, P_test, k=5)

print(f"Eval rows (>=1 label): {n_eval}")
print(f"Recall@3(any-hit): {r_any_3:.4f} | Precision@3: {p3:.4f} | Recall@3: {r3:.4f}")
print(f"Recall@5(any-hit): {r_any_5:.4f} | Precision@5: {p5:.4f} | Recall@5: {r5:.4f}")


train: (4000, 768) (4000, 46)
test : (1000, 768) (1000, 46)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

P_test: (1000, 46)
Eval rows (>=1 label): 1000
Recall@3(any-hit): 0.1510 | Precision@3: 0.0567 | Recall@3: 0.0607
Recall@5(any-hit): 0.2760 | Precision@5: 0.0676 | Recall@5: 0.1297


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
